# Subscription Service Financial Helper

### Step 1: Understand the Problem

**Problem**

Subscription-based services are commonly used for many types of products and have only become more abundant. Individually they can seem small but they can add up quickly especially with differing price amounts and different intervals of recurring payments. This can make it hard for a user to really understand just how much they are spending from these subscriptions, as looking at indidivual payments does not give a clear picture.

**Proposed Solution**

The aim of this project is to create a personal finance assistant that helps users track and analyse their subsciption expenses.

The application will allows users to provide information about their subscriptions like; subscription name, payment amount, frequency, and payment date. This information will be accepted both manually and loaded through a CSV file.

Analysis will then be conducted on the user data giving personalized information and feedback to the user based on their data. For example, a function called combined_costs will calculate the estimated monthly and yearly combined cost of all active subscriptions. There will also be a chatbot personalized to talk to the user which will have its own functions.

**Target Audience**

The target users for this application are people that pay for multiple subscription services and want a simple way to really see how much they are spending and monitor those combined spendings. The application is designed for users who are non-technical and have little financial knowledge, presenting the information in a clear and understandable way.

**Main Goal**

The main goal for this application is to help the user answer the questions, "How much am I really spending on my subscriptions?".

### Step 2: Inputs and Outputs

**Inputs**

- String `name` - User's name
- String `subscription_name` - Name of subscription being entered
- Float `payment_amount` - Cost of one recurring payment
- String `frequency` - Payment interval selected by user
- String `last_payment_date` - Date of most recent payment
- Integer `custom_interval` - Number of days between payments
- File `csv_file` - File with multiple subscriptions
- String `chatbot_input` - Message from user top chatbot

**Outputs**

- `monthly_cost` - Estimated combined monthly cost of all active subscriptions
- `yearly_cost` - Estimated combined monthly cost of all active subscriptions
- `next_payment_date` - Estimated next payment date for a subscription
- `subscription_summary` - Summary of the user's subscriptions and calculated spending information
- `chatbot_response` - Personalised response generated by the finance assistant

### Step 3: Worked Examples

**Example 1: Monthly Subscription**\
User adds subscription that costs \$15 a month\
Monthly cost = $15, Yearly cost = 15 x 12 = $180\
Expected result: monthly_cost = 15.00, yearly_cost = 180.00

**Example 2: Weekly Subscription**\
User adds subscription that costs \$8 a week\
Yearly cost = 8 x 52 = \$416, Monthly cost = 416 / 12 = $34.67\
Expected result: monthly_cost = 34.67, yearly_cost = 416.00

**Example 3: Combined Subscription**\
User puts in csv file with three subscriptions: \$15 per month, \$12 per month, $120 per year.\
Convert all subscriptions to yearly costs: 15 x 12 = $180, 12 x 12 = \$144, \$120\
Combined yearly cost: 180 + 144 + 120 = \$444\
Estimated combined monthly cost: 444 / 12 = \$37\
Expected result: monthly_cost = 37.00, yearly_cost = 444.00

**Example 4: Next Payment Date**\
A subscription was last paid on 1 September 2026 and is charged every month. Next payment should occur one month later 1 October 2026.\
Expected result: next_payment_date = 01/10/2026\
If changed to weekly payments, expected result should adjust accordingly to: next_payment_date = 08/09/2026


**Example 5: Invalid Input**\
User inputs payment amount of a subscription as -20\
payment_amount = -20.00\
This should cause an error and reply with: "Payment amount must be greater than $0."

### Step 4: Pseudocode

The main tasks of the app will be seperated into functions. These functions will be connected to and used through a Gradio interface, below is pseudocode for each task to be designed.

**calculate_cost**\
Purpose: Convert the cost of one subscription into estimated monthly and yearly costs.
```text
FUNCTION calculate_cost(payment_amount, frequency, custom_interval)

    IF payment_amount is less than or equal to 0
        RETURN error

    CASE frequency

        "weekly":
            yearly_cost = payment_amount * 52

        "fortnightly":
            yearly_cost = payment_amount * 26

        "monthly":
            yearly_cost = payment_amount * 12

        "yearly":
            yearly_cost = payment_amount

        "custom":
            IF custom_interval is less than or equal to 0
                RETURN error
            END IF

            payments_per_year = 365 / custom_interval
            yearly_cost = payment_amount * payments_per_year

        DEFAULT:
            RETURN error
    
    END CASE

    monthly_cost = yearly_cost / 12

    RETURN monthly_cost, yearly_cost
END FUNCTION
```

**combined_cost**\
Purpose: Calculate the estimated total monthly and yearly cost of all subscriptions.
```text
FUNCTION combined_cost(subscriptions)

    total_monthly_cost = 0
    total_yearly_cost = 0

    FOR each subscription in subscriptions
        CALL calculate_cost using the subscription payment amount and frequency

        total_monthly_cost += monthly_cost
        total_yearly_cost += yearly_cost

    END FOR

    RETURN total_monthly_cost, total_yearly_cost
END FUNCTION
```
**next_payment**\
Purpose: Estimate when a subscription will next charge the user.
```text
FUNCTION next_payment(last_payment_date, frequency, custom_interval)

    IF last_payment_date is invalid
        RETURN error

    CASE frequency

        "weekly":
            next_payment_date = last_payment_date + 7 days

        "fortnightly":
            next_payment_date = last_payment_date + 14 days

        "monthly":
            next_payment_date = last_payment_date + 1 month

        "yearly":
            next_payment_date = last_payment_date + 1 year

        "custom":
            IF custom_interval is less than or equal to 0
                RETURN error
            END IF

            next_payment_date = last_payment_date + custom_interval days

        DEFAULT:
            RETURN error

    END CASE

    RETURN next_payment_date
END FUNCTION
```
**load_csv**\
Purpose: Load multiple subscriptions from a CSV file and make the information available to the rest of the application.
```text
FUNCTION load_csv(csv_file)

    IF no file was provided
        RETURN error

    TRY
        READ csv_file using pandas

        CHECK that required columns exist:
            subscription_name
            payment_amount
            frequency
            last_payment_date

        IF required columns are missing
            RETURN error

        FOR each subscription in the CSV
            CHECK that payment amount and frequency are valid

        RETURN loaded subscription data

    IF the file cannot be read
        RETURN error
END FUNCTION
```
**calculate_savings**\
Purpose: Calculate how much the user could save by cancelling selected subscriptions.
```text
FUNCTION calculate_savings(selected_subscriptions)

    saved_monthly_cost = 0
    saved_yearly_cost = 0

    IF no subscritpions are selected
        RETURN monthly_savings = 0, yearly_savings = 0
    
    FOR each selected subscription
        CALL calculate_cost with subscription information

        saved_monthly_cost += monthly_cost
        saved_yearly_cost += yearly_cost
    
    END FOR

    RETURN saved_monthly_cost, saved_yearly_cost
END FUNCTION
```
**spending_percentages**\
Purpose: Calculate what percentage of the user's total subscription spending comes from each subscription.
```text
FUNCTION spending_percentages(subscriptions)

    CALL combined_cost to get total_yearly_cost

    IF total_yearly_cost equals 0
        RETURN no percentage data

    CREATE empty results

    FOR each subscription in subscriptions
        CALL calculate_cost for the subscription

        percentage = subscription yearly cost / total_yearly_cost * 100

        STORE subscription name and percentage in results

    END FOR

    RETURN results
END FUNCTION
```